<a href="https://colab.research.google.com/github/denisejroth/bags-vectors-transformers/blob/main/bonus/notebooks/6_neural_network_basics_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Bonus: Neural Networks, the Basics · Solutions

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every ✏️ exercise is filled in with one possible answer and a
short comment. Your numbers will differ a little from run to run, so read the comments as "what
to look for", not as exact results.

Every model on Days 3 and 4 (BERT, sentence transformers, LLMs) is a neural network. This
bonus notebook goes back to the beginning: what a single artificial neuron does, why stacking
them into layers helps, and how a network learns from its mistakes. Then we build **the same
network three times**, in **scikit-learn**, **Keras** and **PyTorch**, so you can see what
each of these "stacks" asks of you and what it gives you in return.

**Contents**

1. One neuron
2. Activation functions
3. Why we need hidden layers: a network from scratch in NumPy
4. Letting the computer do the calculus: PyTorch's autograd
5. Three stacks, one model
6. The data, and a baseline
7. Stack 1: scikit-learn
8. Stack 2: Keras
9. Stack 3: PyTorch
10. Comparing the stacks
11. Overfitting, and how to fight it
12. When are neural networks worth it for text?

> A GPU is **not** needed for this notebook. Everything runs on Colab's free CPU in a few minutes.

## 0. Setup

Colab comes with all three libraries installed, so there's nothing to install.

One choice deserves a word. **Keras 3** is a front end that can run on three different engines
(*backends*): TensorFlow, JAX or PyTorch. We run it on **PyTorch**. That way, Keras and PyTorch
share one engine in this notebook and don't compete for memory, and the Keras code itself is
exactly the same as with TensorFlow. The backend has to be chosen **before** Keras is imported,
which is what the first line does. (To try TensorFlow instead, change it to `"tensorflow"` and
restart the session with *Runtime → Restart session*.)

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"     # must come before `import keras`

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import keras

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

# Each library has its own random number generator, so we seed them all
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
keras.utils.set_random_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 100)

print("PyTorch", torch.__version__)
print("Keras  ", keras.__version__, "running on", keras.backend.backend())
print("Device ", device.upper())

## 1. One neuron

An artificial neuron does three things:

1. It multiplies each input by a **weight** (how much that input matters, and in which direction).
2. It adds everything up, plus a **bias** (a baseline it starts from).
3. It passes the sum through an **activation function**, which decides what comes out.

In formula form: $\text{output} = f(w_1x_1 + w_2x_2 + \dots + w_nx_n + b)$.

Imagine a neuron that reads three word counts from a bill title, for *tax*, *school* and *credit*:

In [ ]:
x = np.array([1.0, 0.0, 2.0])     # inputs: "tax" once, "school" never, "credit" twice
w = np.array([0.8, -0.5, 0.3])    # weights: "tax" and "credit" push up, "school" pushes down
b = -0.2                          # bias

z = np.dot(w, x) + b              # step 1 + 2: weighted sum plus bias
output = 1 / (1 + np.exp(-z))     # step 3: the sigmoid squeezes z into a number between 0 and 1

print(f"weighted sum z = {z:.2f}")
print(f"output         = {output:.3f}")

If that looks familiar, it should: **a single neuron with a sigmoid activation is exactly logistic
regression**, the classifier from Day 1. The output reads as a probability (here: "how likely is
this a tax bill?"), and the weights play the role of the coefficients.

So where does the "neural network" magic come from? From two things: other **activation
functions**, and connecting many neurons into **layers**.

> **✏️ Exercise 1**
>
> 1. Compute the output of the same neuron for a title that mentions *school* three times and
>    nothing else (`x = [0, 3, 0]`). Is the output closer to 0 or to 1? Why?
> 2. Replace the sigmoid with a **ReLU**: `max(0, z)`. What comes out for both inputs?

In [ ]:
# ✅ Solution
x_school = np.array([0.0, 3.0, 0.0])
z_school = np.dot(w, x_school) + b
print(f"z = {z_school:.2f}, sigmoid output = {1 / (1 + np.exp(-z_school)):.3f}")
# z = 3 * (-0.5) - 0.2 = -1.7. A negative sum gives a sigmoid output below 0.5 (about 0.15):
# "school" has a negative weight, so it pushes this neuron away from "tax bill".

relu = lambda v: max(0.0, v)
print(f"ReLU output, tax-and-credit title: {relu(z):.2f}")
print(f"ReLU output, school title:         {relu(z_school):.2f}")
# ReLU passes positive sums through unchanged and turns negative ones into 0.

## 2. Activation functions

The activation function is what lets a network learn patterns that aren't straight lines. The
three you'll meet most often:

In [ ]:
z = np.linspace(-5, 5, 200)
activations = {
    "sigmoid": 1 / (1 + np.exp(-z)),
    "tanh": np.tanh(z),
    "ReLU": np.maximum(0, z),
}

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharex=True)
for ax, (name, values) in zip(axes, activations.items()):
    ax.plot(z, values, color="#34B233", linewidth=2.5)
    ax.axhline(0, color="gray", linewidth=0.5)
    ax.axvline(0, color="gray", linewidth=0.5)
    ax.set_title(name)
    ax.set_xlabel("z (weighted sum)")
plt.tight_layout()
plt.show()

| Activation | Output range | Where you'll see it |
|---|---|---|
| **sigmoid** | 0 to 1 | the output of a yes/no classifier (a probability) |
| **tanh** | -1 to 1 | inside older networks, such as the LSTMs in Bonus 8 |
| **ReLU** | 0 to ∞ | hidden layers of most modern networks: simple, fast, and it trains well |

For more than two classes, the output layer uses **softmax**, which turns one score per class
into probabilities that add up to 1. You met it on Day 3.

**Why not leave the activation out?** Without a non-linear activation, stacking layers is
pointless: a linear function of a linear function is still linear, so ten layers would be no
more expressive than one.

## 3. Why we need hidden layers: a network from scratch

Here is the classic problem that a single neuron cannot solve, called **XOR** ("exclusive or").
The output should be 1 when *exactly one* of the two inputs is 1:

| $x_1$ | $x_2$ | XOR |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

A text version: a bill is about *one* topic or the other, but a title that mentions both is
probably about something else. No single straight line can separate the 1s from the 0s, so
logistic regression (one neuron) is stuck:

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

single_neuron = LogisticRegression().fit(X_xor, y_xor)
print("Logistic regression predicts:", single_neuron.predict(X_xor), " (should be [0 1 1 0])")
print("Accuracy:", accuracy_score(y_xor, single_neuron.predict(X_xor)))

Now we add a **hidden layer** of four neurons between the inputs and the output, and train the
network ourselves, in plain NumPy. Training repeats four steps, many times:

1. **Forward pass**: push the inputs through the network to get predictions.
2. **Loss**: measure how wrong the predictions are. We use *binary cross-entropy*, which punishes
   confident mistakes heavily.
3. **Backward pass** (*backpropagation*): use the chain rule from calculus to work out, for every
   weight, in which direction it should move to make the loss smaller. This is the *gradient*.
4. **Update**: move every weight a small step in that direction. The size of the step is the
   **learning rate**. This is *gradient descent*.

You don't need to follow every line of the backward pass. The point is to see that there is no
magic, only arithmetic.

In [ ]:
def sigmoid(v):
    return 1 / (1 + np.exp(-v))


def train_xor(n_hidden=4, learning_rate=0.5, epochs=5000, seed=SEED):
    rng = np.random.default_rng(seed)
    W1 = rng.normal(0, 1, (2, n_hidden))     # weights input -> hidden
    b1 = np.zeros(n_hidden)
    W2 = rng.normal(0, 1, (n_hidden, 1))     # weights hidden -> output
    b2 = np.zeros(1)
    y = y_xor.astype(float)
    losses = []

    for epoch in range(epochs):
        # 1. Forward pass
        h = np.tanh(X_xor @ W1 + b1)                     # hidden layer activations, shape (4, n_hidden)
        p = sigmoid(h @ W2 + b2).ravel()                 # predicted probabilities, shape (4,)
        p = np.clip(p, 1e-7, 1 - 1e-7)                   # avoid log(0)

        # 2. Loss: binary cross-entropy
        loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
        losses.append(loss)

        # 3. Backward pass: the chain rule, layer by layer, from the output back to the input
        d_out = (p - y)[:, None] / len(y)                # gradient at the output
        dW2, db2 = h.T @ d_out, d_out.sum(axis=0)
        d_hidden = (d_out @ W2.T) * (1 - h ** 2)         # back through tanh (its slope is 1 - tanh^2)
        dW1, db1 = X_xor.T @ d_hidden, d_hidden.sum(axis=0)

        # 4. Update: a small step against the gradient
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

    return p, losses


p_xor, losses = train_xor()
print("Predicted probabilities:", p_xor.round(3))
print("Predicted classes:      ", (p_xor > 0.5).astype(int), " (should be [0 1 1 0])")

plt.figure(figsize=(6, 3))
plt.plot(losses, color="#34B233")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("A tiny network learning XOR")
plt.show()

The hidden layer gives the network an intermediate representation in which the problem *does*
become separable. That's the core idea of deep learning in one sentence: **hidden layers learn
new representations of the input**, and the last layer only has to draw a simple boundary in
that new space. A word embedding (Day 2) is exactly such a learned representation.

> **✏️ Exercise 2**
>
> Use `train_xor` to explore what the network needs:
>
> 1. Train with `n_hidden=1` and with `n_hidden=2`. Does it still solve XOR?
> 2. Train with `learning_rate=0.01` and with `learning_rate=5`. Plot the four loss curves (two
>    sizes, two learning rates) in one figure. What goes wrong with each learning rate?

In [ ]:
# ✅ Solution
for n_hidden in [1, 2, 4]:
    p, _ = train_xor(n_hidden=n_hidden)
    print(f"n_hidden={n_hidden}: predicted {(p > 0.5).astype(int)}")
# One hidden neuron can't do it: that's back to a single boundary. Two hidden neurons are the
# theoretical minimum for XOR, but with an unlucky start they can get stuck. Four make it easy.

plt.figure(figsize=(7, 3.5))
for label, kwargs in {"1 hidden": dict(n_hidden=1), "2 hidden": dict(n_hidden=2),
                      "lr=0.01": dict(learning_rate=0.01), "lr=5": dict(learning_rate=5)}.items():
    _, curve = train_xor(**kwargs)
    plt.plot(curve, label=label)
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.show()
# lr=0.01: steps are so small that the loss barely moves in 5000 epochs (too slow).
# lr=5: steps are so big that the network overshoots, and the loss jumps around or gets stuck.

## 4. Letting the computer do the calculus

Writing the backward pass by hand works for a toy network. For BERT, with 66 million weights
in dozens of layers, it would be hopeless. The most important thing deep learning frameworks do
is **automatic differentiation** (*autograd*): you only write the forward pass, and the
framework works out all the gradients for you.

Here's the same XOR network in PyTorch. Compare it with the NumPy version: the backward pass is
now a single line, `loss.backward()`.

In [ ]:
torch.manual_seed(SEED)

Xt = torch.tensor(X_xor, dtype=torch.float32)
yt = torch.tensor(y_xor, dtype=torch.float32)

net = nn.Sequential(nn.Linear(2, 4), nn.Tanh(), nn.Linear(4, 1))   # input -> 4 hidden -> output
loss_fn = nn.BCEWithLogitsLoss()                                   # sigmoid + binary cross-entropy
optimizer = torch.optim.SGD(net.parameters(), lr=0.5)

for epoch in range(5000):
    logits = net(Xt).squeeze(1)      # 1. forward pass
    loss = loss_fn(logits, yt)       # 2. loss
    optimizer.zero_grad()            #    (clear the gradients from the previous step)
    loss.backward()                  # 3. backward pass: PyTorch computes every gradient
    optimizer.step()                 # 4. update the weights

print("Predicted classes:", (torch.sigmoid(net(Xt)).squeeze(1) > 0.5).int().tolist())
print(f"Final loss: {loss.item():.4f}")

Those four lines inside the loop are the heartbeat of every neural network you'll ever train,
including the `Trainer` on Day 3, which runs exactly this loop for you behind the scenes.

**The vocabulary of training**, in one place:

| Term | Meaning |
|---|---|
| **epoch** | one pass over the whole training set |
| **batch** | the group of examples processed before each weight update |
| **loss** | a number measuring how wrong the predictions are; training tries to make it small |
| **gradient** | for each weight, the direction (and steepness) in which the loss increases |
| **optimizer** | the rule for updating the weights from the gradients (SGD, Adam, ...) |
| **learning rate** | how big each update step is |

## 5. Three stacks, one model

From here on we build a real text classifier, and we build it three times. All three stacks
can express the same simple network, but they make very different trade-offs:

| | **scikit-learn** | **Keras** | **PyTorch** |
|---|---|---|---|
| Style | `fit` / `predict`, like every other scikit-learn model | stack layers, `compile`, `fit` | write the model class **and** the training loop yourself |
| What it can build | only simple feed-forward networks | almost anything | anything |
| GPU support | no | yes | yes |
| Strength | fastest way to try a neural network on a baseline | concise and readable; good defaults | full control; the standard in research |
| You'll meet it in | quick comparisons | many tutorials and applied work | Hugging Face `transformers` (Day 3), most new research code |

Two more names you'll come across. **TensorFlow** is Google's framework and the classic engine
underneath Keras; Keras 3 can also run on **JAX** (Google's research framework) or, as in this
notebook, on PyTorch. And Hugging Face's `Trainer` from Day 3 is a Keras-like convenience layer
on top of PyTorch.

## The data: US Congressional bills (again)

We use the same data and the **same split** as the Day 3 BERT notebook: bill titles from
`dreamproit/bill_labels_us`, four policy areas (Health, Education, Taxation, Armed Forces and
National Security), 500 bills per area, split 70/15/15 into training, validation and test sets
with the same random seed. That means you can put the scores from this notebook right next to
the TF-IDF baseline and the fine-tuned BERT model from Day 3.

(As on Day 3, we read only two ~200 MB chunks of the dataset's Parquet copy, and only the
columns we need, because the full dataset is a single 5 GB file.)

In [ ]:
from huggingface_hub import hf_hub_download
from sklearn.model_selection import train_test_split

parts = []
for c in [0, 5]:
    path = hf_hub_download(repo_id="dreamproit/bill_labels_us", repo_type="dataset",
                           revision="refs/convert/parquet", filename=f"default/train/{c:04d}.parquet")
    parts.append(pd.read_parquet(path, columns=["title", "policy_area"]))
raw = pd.concat(parts, ignore_index=True)

AREAS = ["Health", "Education", "Taxation", "Armed Forces and National Security"]
clean = (raw.assign(title=raw["title"].fillna("").str.strip())
            .query("title != ''")
            .drop_duplicates(subset="title"))                 # every title once: no leakage
bills = (clean[clean["policy_area"].isin(AREAS)]
         .sample(frac=1, random_state=SEED)
         .groupby("policy_area").head(500)
         .reset_index(drop=True))

labels = sorted(AREAS)
label2id = {name: i for i, name in enumerate(labels)}
bills["label"] = bills["policy_area"].map(label2id)

train_df, temp_df = train_test_split(bills, test_size=0.30, stratify=bills["label"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)
y_train, y_val, y_test = train_df["label"].values, val_df["label"].values, test_df["label"].values

print(f"train: {len(train_df)}   validation: {len(val_df)}   test: {len(test_df)}")
print("Label IDs:", label2id)

### Features: TF-IDF

To keep the comparison about the *models*, all three stacks get exactly the same input: the
TF-IDF features from Day 1 and Day 3. Preparing the data is framework-independent; only the
model code differs. Neural network libraries want dense arrays of 32-bit floats, so we convert.

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train = tfidf.fit_transform(train_df["title"]).toarray().astype("float32")
X_val = tfidf.transform(val_df["title"]).toarray().astype("float32")
X_test = tfidf.transform(test_df["title"]).toarray().astype("float32")

n_features, n_classes = X_train.shape[1], len(labels)
print(f"{X_train.shape[0]} training titles, {n_features} TF-IDF features, {n_classes} classes")

And the baseline every network has to beat, logistic regression (the Day 3 baseline). We keep
all validation scores in one dictionary.

In [ ]:
results = {}

def score(name, y_true, y_pred):
    results[name] = {"accuracy": accuracy_score(y_true, y_pred),
                     "macro-F1": f1_score(y_true, y_pred, average="macro")}
    print(f"{name:<28} validation accuracy {results[name]['accuracy']:.3f} | "
          f"macro-F1 {results[name]['macro-F1']:.3f}")

baseline = LogisticRegression(max_iter=1000).fit(X_train, y_train)
score("Logistic regression", y_val, baseline.predict(X_val))

## 7. Stack 1: scikit-learn

`MLPClassifier` (MLP = *multi-layer perceptron*, the classic name for a feed-forward network)
behaves like any other scikit-learn model. Our network has one hidden layer of 64 ReLU neurons.
`early_stopping=True` sets aside 10% of the training data and stops when the score on it stops
improving (Section 11 explains why).

In [ ]:
start = time.time()
sk_mlp = MLPClassifier(
    hidden_layer_sizes=(64,),   # one hidden layer with 64 neurons
    activation="relu",
    alpha=1e-4,                 # L2 regularization (keeps weights small)
    learning_rate_init=1e-3,
    early_stopping=True,
    max_iter=200,
    random_state=SEED,
)
sk_mlp.fit(X_train, y_train)
print(f"Trained for {sk_mlp.n_iter_} epochs in {time.time() - start:.1f} s")
score("scikit-learn MLP", y_val, sk_mlp.predict(X_val))

Four lines of real code. The price of that convenience: no GPU, and no way to build anything
other than stacked fully connected layers. No convolutions (Bonus 7), no recurrence (Bonus 8).

> **✏️ Exercise 3**
>
> Try three other architectures with `MLPClassifier`: `hidden_layer_sizes=(16,)`, `(256,)` and
> `(64, 64)` (two hidden layers). Compare their validation macro-F1 with the 64-neuron network.
> Does bigger mean better here?

In [ ]:
# ✅ Solution
for sizes in [(16,), (64,), (256,), (64, 64)]:
    m = MLPClassifier(hidden_layer_sizes=sizes, activation="relu", early_stopping=True,
                      max_iter=200, random_state=SEED).fit(X_train, y_train)
    f1 = f1_score(y_val, m.predict(X_val), average="macro")
    print(f"hidden layers {str(sizes):<10} validation macro-F1 {f1:.3f}")
# On short titles with TF-IDF features, size usually matters little: the signal is mostly in
# a handful of keywords, which even a small network (or logistic regression) picks up.

## 8. Stack 2: Keras

In Keras you describe the network as a stack of layers, then `compile` it (choose the optimizer,
the loss and the metrics) and `fit` it. Same architecture as before, plus **dropout**: during
training it randomly switches off half of the hidden neurons at every step, so the network
can't lean too heavily on any single one (more in Section 11).

In [ ]:
keras.utils.set_random_seed(SEED)

keras_model = keras.Sequential([
    keras.Input(shape=(n_features,)),
    keras.layers.Dense(64, activation="relu"),          # hidden layer
    keras.layers.Dropout(0.5),                          # dropout: only active during training
    keras.layers.Dense(n_classes, activation="softmax") # output: one probability per class
])
keras_model.summary()

The summary counts the **parameters**: every weight and bias the network will learn. The first
layer has one weight per (feature, neuron) pair plus one bias per neuron, which is why it holds
almost all of them.

In [ ]:
keras_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),   # Adam: the go-to optimizer
    loss="sparse_categorical_crossentropy",                # "sparse" = labels are integers 0..3
    metrics=["accuracy"],
)

start = time.time()
history = keras_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    verbose=0,                       # set to 1 or 2 to see progress per epoch
)
print(f"Trained 20 epochs in {time.time() - start:.1f} s")

keras_pred = keras_model.predict(X_val, verbose=0).argmax(axis=1)   # most likely class per title
score("Keras MLP", y_val, keras_pred)

In [ ]:
def plot_history(train_loss, val_loss, title):
    plt.figure(figsize=(6.5, 3.2))
    plt.plot(train_loss, label="training loss", color="#1A1A2E")
    plt.plot(val_loss, label="validation loss", color="#34B233")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title(title); plt.legend()
    plt.show()

plot_history(history.history["loss"], history.history["val_loss"], "Keras MLP")

> **✏️ Exercise 4**
>
> Build a second Keras model with **two** hidden layers of 64 neurons and a dropout of 0.3 after
> each. Train it the same way and compare its validation macro-F1 and its loss curves with the
> first model.

In [ ]:
# ✅ Solution
keras.utils.set_random_seed(SEED)
deeper = keras.Sequential([
    keras.Input(shape=(n_features,)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(n_classes, activation="softmax"),
])
deeper.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h = deeper.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=20, batch_size=32, verbose=0)

pred = deeper.predict(X_val, verbose=0).argmax(axis=1)
print(f"Two hidden layers: validation macro-F1 {f1_score(y_val, pred, average='macro'):.3f}")
plot_history(h.history["loss"], h.history["val_loss"], "Keras, two hidden layers")
# Expect similar scores. With less dropout and more parameters, the validation loss often
# starts rising earlier: a sign of overfitting (Section 11).

## 9. Stack 3: PyTorch

In PyTorch you write two things yourself: the **model** (a class with a `forward` method that
says how inputs flow through the layers) and the **training loop** (the four steps from
Section 4, wrapped in loops over epochs and batches). More code, but nothing is hidden, and
that's exactly why researchers like it.

### The model

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_in, n_hidden, n_out, dropout=0.5):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_in, n_hidden),    # hidden layer
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(n_hidden, n_out),   # output layer: one raw score (logit) per class
        )

    def forward(self, x):
        return self.layers(x)             # no softmax here: the loss function applies it


torch.manual_seed(SEED)
torch_model = MLP(n_features, 64, n_classes).to(device)
print(torch_model)
print("Parameters:", sum(p.numel() for p in torch_model.parameters()))

Same number of parameters as the Keras model: it really is the same network. One difference to
notice: the Keras model ends in a softmax, while the PyTorch model outputs raw scores (logits).
PyTorch's `CrossEntropyLoss` applies the softmax internally, which is numerically more stable.

### The data loaders

A `DataLoader` serves the data in shuffled batches.

In [ ]:
def make_loader(X, y, shuffle):
    dataset = TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.long))
    return DataLoader(dataset, batch_size=32, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)    # reshuffled every epoch
val_loader = make_loader(X_val, y_val, shuffle=False)

### The training loop

`model.train()` and `model.eval()` switch dropout on and off. `torch.no_grad()` skips the
gradient bookkeeping when we only evaluate.

In [ ]:
def evaluate_torch(model, loader, loss_fn):
    model.eval()                                   # dropout off
    total_loss, all_preds = 0.0, []
    with torch.no_grad():                          # no gradients needed for evaluation
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total_loss += loss_fn(logits, yb).item() * len(yb)
            all_preds.append(logits.argmax(dim=1).cpu())
    return total_loss / len(loader.dataset), torch.cat(all_preds).numpy()


def train_torch(model, epochs=20, lr=1e-3):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()                              # dropout on
        running = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)                     # 1. forward pass
            loss = loss_fn(logits, yb)             # 2. loss
            optimizer.zero_grad()
            loss.backward()                        # 3. backward pass
            optimizer.step()                       # 4. update
            running += loss.item() * len(yb)
        val_loss, _ = evaluate_torch(model, val_loader, loss_fn)
        history["loss"].append(running / len(train_loader.dataset))
        history["val_loss"].append(val_loss)
    return history


start = time.time()
torch_history = train_torch(torch_model)
print(f"Trained 20 epochs in {time.time() - start:.1f} s")

_, torch_pred = evaluate_torch(torch_model, val_loader, nn.CrossEntropyLoss())
score("PyTorch MLP", y_val, torch_pred)
plot_history(torch_history["loss"], torch_history["val_loss"], "PyTorch MLP")

> **✏️ Exercise 5**
>
> Keras and scikit-learn stop training early for you if you ask them to; in PyTorch you write it
> yourself. Write `train_torch_early_stopping(model, max_epochs=50, patience=3)`: after every
> epoch, check the validation loss. Keep a copy of the model's weights whenever it reaches a new
> best, stop when it hasn't improved for `patience` epochs, and restore the best weights at the
> end.
>
> *Hint: `copy.deepcopy(model.state_dict())` saves the weights, `model.load_state_dict(...)`
> restores them.*

In [ ]:
# ✅ Solution
import copy

def train_torch_early_stopping(model, max_epochs=50, patience=3, lr=1e-3):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_loss, best_state, epochs_without_improvement = float("inf"), None, 0

    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        val_loss, _ = evaluate_torch(model, val_loader, loss_fn)
        if val_loss < best_loss:
            best_loss, best_state = val_loss, copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Stopping after epoch {epoch + 1}; best validation loss {best_loss:.4f}")
                break

    model.load_state_dict(best_state)
    return model


torch.manual_seed(SEED)
es_model = train_torch_early_stopping(MLP(n_features, 64, n_classes).to(device))
_, es_pred = evaluate_torch(es_model, val_loader, nn.CrossEntropyLoss())
print(f"With early stopping: validation macro-F1 {f1_score(y_val, es_pred, average='macro'):.3f}")

## 10. Comparing the stacks

In [ ]:
pd.DataFrame(results).T.round(3)

The three networks are the same model on paper, but their scores differ a little, for reasons
that have nothing to do with which stack is "better": different starting weights, slightly
different optimizer defaults, and scikit-learn's early stopping using only 90% of the training
data. Differences of a point or so are noise (Exercise 6 shows how much).

Here is how the same ideas are spelled in each stack:

| Idea | scikit-learn | Keras | PyTorch |
|---|---|---|---|
| hidden layer | `hidden_layer_sizes=(64,)` | `Dense(64, activation="relu")` | `nn.Linear(n_in, 64)` + `nn.ReLU()` |
| dropout | (not available) | `Dropout(0.5)` | `nn.Dropout(0.5)` |
| loss | built in | `loss="sparse_categorical_crossentropy"` | `nn.CrossEntropyLoss()` |
| optimizer | `solver="adam"` (default) | `optimizer=Adam(...)` | `torch.optim.Adam(...)` |
| training | `.fit(X, y)` | `.fit(X, y, epochs=..., batch_size=...)` | your own loop |
| predicting | `.predict(X)` | `.predict(X).argmax(axis=1)` | `model(X).argmax(dim=1)` |

Which one to choose? For a quick check of whether a neural network helps at all,
scikit-learn. For building something custom without writing loops, Keras. For working with
Hugging Face models or adapting research code, PyTorch. Most people end up reading all three.

> **✏️ Exercise 6**
>
> How much does the Keras result move if only the random seed changes? Train the original Keras
> model (64 hidden neurons, dropout 0.5, 20 epochs) with seeds 1 to 5 and report the mean and
> standard deviation of the validation macro-F1. Are the differences between the stacks larger
> than that?

In [ ]:
# ✅ Solution
scores = []
for s in range(1, 6):
    keras.utils.set_random_seed(s)
    m = keras.Sequential([
        keras.Input(shape=(n_features,)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(0.5),
        keras.layers.Dense(n_classes, activation="softmax"),
    ])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
    m.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
    scores.append(f1_score(y_val, m.predict(X_val, verbose=0).argmax(axis=1), average="macro"))

print("Macro-F1 per seed:", np.round(scores, 3))
print(f"Mean {np.mean(scores):.3f}, standard deviation {np.std(scores, ddof=1):.3f}")
# If the gaps between the stacks are about the size of this standard deviation, they are not
# meaningful: same lesson as the seeds section of the Day 3 notebook.

## 11. Overfitting, and how to fight it

Neural networks have many parameters, which makes them very good at memorizing their training
data, including its noise. Let's provoke that on purpose: a big network, no dropout, many epochs.

In [ ]:
keras.utils.set_random_seed(SEED)
big = keras.Sequential([
    keras.Input(shape=(n_features,)),
    keras.layers.Dense(512, activation="relu"),
    keras.layers.Dense(512, activation="relu"),
    keras.layers.Dense(n_classes, activation="softmax"),
])
big.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
big_history = big.fit(X_train, y_train, validation_data=(X_val, y_val),
                      epochs=40, batch_size=32, verbose=0)

plot_history(big_history.history["loss"], big_history.history["val_loss"],
             "Big network, no dropout: overfitting")
print(f"Final training accuracy:   {big_history.history['accuracy'][-1]:.3f}")
print(f"Final validation accuracy: {big_history.history['val_accuracy'][-1]:.3f}")

The training loss keeps falling toward zero, while the validation loss bottoms out early and
then climbs: the network is getting *more confident* on titles it has memorized and *worse* on
new ones. Three standard remedies:

- **Dropout**: randomly silence neurons during training, so no single neuron becomes essential.
- **Weight decay / L2 regularization**: penalize large weights (`alpha` in scikit-learn,
  `weight_decay` in PyTorch optimizers and in the Day 3 `TrainingArguments`).
- **Early stopping**: watch the validation loss and stop when it stops improving, keeping the
  best weights. In Keras that's a *callback*:

In [ ]:
keras.utils.set_random_seed(SEED)
big_es = keras.Sequential([
    keras.Input(shape=(n_features,)),
    keras.layers.Dense(512, activation="relu"),
    keras.layers.Dense(512, activation="relu"),
    keras.layers.Dense(n_classes, activation="softmax"),
])
big_es.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
es_history = big_es.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=40, batch_size=32, callbacks=[early_stop], verbose=0)

print(f"Stopped after {len(es_history.history['loss'])} of 40 epochs")
score("Keras big + early stopping", y_val, big_es.predict(X_val, verbose=0).argmax(axis=1))

### The test set, once

We've made all our choices on the validation set, so we can now look at the test set, which is
the same 300 titles as in Day 3. Compare these numbers with the TF-IDF baseline and the
fine-tuned DistilBERT from the Day 3 notebook.

In [ ]:
_, torch_test_pred = evaluate_torch(torch_model, make_loader(X_test, y_test, False), nn.CrossEntropyLoss())
test_scores = {
    "Logistic regression": baseline.predict(X_test),
    "scikit-learn MLP": sk_mlp.predict(X_test),
    "Keras MLP": keras_model.predict(X_test, verbose=0).argmax(axis=1),
    "PyTorch MLP": torch_test_pred,
}
pd.DataFrame({name: {"test accuracy": accuracy_score(y_test, p),
                     "test macro-F1": f1_score(y_test, p, average="macro")}
              for name, p in test_scores.items()}).T.round(3)

## 12. When are neural networks worth it for text?

Probably the most useful lesson of this notebook is a sobering one: **on bag-of-words features,
a neural network rarely does much better than logistic regression.** The input already threw
away word order and context, so there is little extra structure for the hidden layers to find.

Neural networks pay off when the *architecture* can exploit something a bag of words can't:

- **Word order and local phrases**: convolutional networks (Bonus 7).
- **Sequences and long-range dependencies**: recurrent networks (Bonus 8).
- **Context and knowledge from huge amounts of text**: pre-trained transformers like BERT
  (Day 3), which combine attention with pre-training.

### Further reading

- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep learning*. MIT Press. Free online at
  [deeplearningbook.org](https://www.deeplearningbook.org).
- Nielsen, M. (2015). *Neural networks and deep learning*. Determination Press. Free online at
  [neuralnetworksanddeeplearning.com](http://neuralnetworksanddeeplearning.com). A gentle,
  hands-on introduction to backpropagation.
- Chollet, F. (2021). *Deep learning with Python* (2nd ed.). Manning. Written by the creator of
  Keras.